
# KDIC RAG V4.7 — 전체 로직 주석·학습용 버전

이 파일은 원본 `KDIC_RAG_V4_7_INTERACTIVE_CHAT.ipynb`의 **실행 로직을 유지하면서**
각 코드 셀 앞에 상세 설명과 코드 내부 주석을 추가한 학습용 사본입니다.

## 전체 흐름을 먼저 한 문장으로 보기

사용자 질문을 숫자 벡터로 바꾸고, 미리 임베딩된 청크 중 의미가 가까운 Top-K를 찾은 다음,
그 청크만 HCX-005에 근거로 제공하여 답변하고 결과를 JSON으로 저장합니다.

```text
KDIC_output.zip
  ├─ documents.jsonl
  ├─ chunks.jsonl
  └─ chunk_embeddings_hcx.jsonl
             ↓ 로드·연결 검사
사용자 질문 → bge-m3 질문 임베딩
             ↓
저장된 청크 벡터와 코사인 유사도 계산
             ↓
업무 필터(선택) → 최소 점수 → Top-K
             ↓
검색 청크를 HCX-005 프롬프트에 삽입
             ↓
답변 생성 → 미지원 URL·전화번호 제거
             ↓
답변 + 근거 청크 + 공식 출처 + 결과 JSON
```

## 꼭 구분해서 이해할 용어

| 용어 | 이 노트북에서 하는 일 |
|---|---|
| 문서(document) | 원문 페이지 단위의 자료 |
| 청크(chunk) | 검색하기 쉽도록 문서를 작게 나눈 텍스트 |
| 임베딩(embedding) | 텍스트 의미를 비교하기 위한 숫자 벡터 |
| Dense 검색 | 질문 벡터와 청크 벡터의 유사도로 검색 |
| 코사인 유사도 | 두 벡터 방향의 유사성을 나타내는 점수 |
| Top-K | 점수가 높은 결과를 최대 몇 개 사용할지 지정 |
| 최소 점수 | 관련성이 너무 낮은 청크를 버리는 기준 |
| RAG | 검색한 근거를 LLM에 함께 주어 답변하게 하는 구조 |
| Grounded answer | 검색 근거 범위 안에서 생성한 답변 |

## 현재 초기형에서 아직 하지 않는 것

이 버전에는 BM25 키워드 검색, Hybrid 검색, Reranking, 업무 자동 분류,
Parent-Child 문맥 확장, 평가셋 일괄 실행, 멀티턴 기억, 웹 UI가 없습니다.
따라서 이 노트북은 완성 서비스가 아니라 **Dense RAG의 기본 동작을 이해하고 검증하는 기준선**입니다.


# KDIC RAG V4.7 — 기존 Output 업로드 기반 대화형 Baseline 답변 생성

이 노트북은 **크롤링·파싱·청킹·문서 임베딩을 다시 실행하지 않습니다.**

기존 V4.6에서 생성한 `KDIC_output.zip`을 업로드한 뒤 다음 파일을 그대로 불러옵니다.

```text
KDIC_output/processed/documents.jsonl
KDIC_output/processed/chunks.jsonl
KDIC_output/processed/chunk_embeddings_hcx.jsonl
```

실행 흐름은 다음과 같습니다.

```text
KDIC_output.zip 업로드
→ 기존 문서·청크·청크 임베딩 로드
→ 입력창에 사용자 질문 직접 작성
→ 질문 1건을 bge-m3로 임베딩
→ 저장된 청크 임베딩과 코사인 유사도 계산
→ Top-K 공식 청크 검색
→ 기존 V4.6 HCX-005 Baseline 프롬프트로 답변 생성
→ 허용되지 않은 URL·전화번호 제거
→ 검색 결과 표·근거 청크·공식 출처 표시
→ 다음 질문 입력
```

> 청크 임베딩은 다시 생성하지 않습니다. 사용자가 입력한 질문만 실행 시점에 임베딩합니다.
>
> 질문 입력창에 `종료`, `끝`, `exit`, `quit` 중 하나를 입력하면 대화형 실행이 끝납니다.


## 1. 필요한 라이브러리 설치

크롤링과 파일 파싱 라이브러리는 설치하지 않습니다. 기존 산출물을 읽고 HCX API를 호출하는 데 필요한 라이브러리만 설치합니다.



### 코드 해설 — 라이브러리 설치

이 셀은 네이버 클로바 스튜디오 HCX API를 **OpenAI 호환 방식**으로 호출할 수 있도록
`openai` 파이썬 패키지를 설치합니다.

- `%pip`: 현재 노트북 커널에 패키지를 설치하는 Jupyter 전용 명령입니다.
- `>=1.68,<2`: 1.68 이상, 2.0 미만 버전만 허용합니다.
- `only-if-needed`: 이미 호환되는 의존성이 있으면 불필요하게 전부 바꾸지 않습니다.
- 이 단계는 모델을 PC에 설치하는 것이 아닙니다. API 요청용 클라이언트만 설치합니다.

실행 결과가 조용한 이유는 `-q(quiet)` 옵션 때문입니다. 설치 후 import 오류가 나면
런타임을 다시 시작하고 다음 셀부터 실행하면 됩니다.


In [ ]:
# 현재 노트북 커널에 HCX API 호출용 openai 패키지를 설치합니다.
%pip -q install --upgrade-strategy only-if-needed "openai>=1.68,<2"


## 2. 공통 설정

- Chat 모델: `HCX-005`
- 질문 임베딩 모델: `bge-m3`
- 기본 검색 개수: `Top-5`
- 기본 최소 유사도: `0.30`

업무 필터를 사용하지 않으려면 `HCX_RAG_BUSINESS_FUNCTION = None`으로 둡니다.



### 코드 해설 — import, 모델 설정, 작업 폴더

이 셀에는 앞으로 모든 함수가 공통으로 사용하는 **환경 설정값**이 모여 있습니다.

1. `import`는 JSONL, ZIP, 경로, 벡터 계산, 표 출력, API 호출 도구를 불러옵니다.
2. `HCX_BASE_URL`은 요청을 보낼 클로바 스튜디오 서버 주소입니다.
3. `HCX_CHAT_MODEL`은 최종 문장 생성 모델, `HCX_EMBEDDING_MODEL`은 질문을 숫자 벡터로 바꾸는 모델입니다.
4. `TOP_K=5`는 유사도가 높은 청크를 최대 5개 가져온다는 뜻입니다.
5. `MIN_SCORE=0.30`은 0.30 미만인 검색 결과를 버리는 1차 기준입니다.
6. `BUSINESS_FUNCTION=None`이면 6개 업무 전체가 검색 대상입니다.
7. `/content/...` 경로는 Google Colab의 임시 저장 공간입니다. 런타임이 종료되면 사라질 수 있습니다.

핵심 구분:

- 임베딩 모델: 질문과 청크의 **의미상 거리 계산**
- 채팅 모델: 검색 근거를 읽고 **사람이 읽을 답변 생성**
- `Top-K`, 최소 점수: 모델이 아니라 **검색기가 사용하는 설정**


In [ ]:
# 타입 힌트에서 현재 클래스명을 바로 참조할 수 있게 평가 시점을 늦춥니다.
from __future__ import annotations

import json
import math
import os
import re
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from openai import OpenAI


# ============================================================
# 모델·검색 Baseline 설정
# ============================================================

HCX_BASE_URL = "https://clovastudio.stream.ntruss.com/v1/openai"
HCX_CHAT_MODEL = "HCX-005"
HCX_EMBEDDING_MODEL = "bge-m3"
HCX_EMBEDDING_ENCODING_FORMAT = "float"

HCX_REQUEST_TIMEOUT_SECONDS = 120
HCX_MAX_RETRIES = 4

HCX_RAG_TOP_K = 5
HCX_RAG_MIN_SCORE = 0.30

# None이면 6개 업무 전체에서 검색합니다.
# 예: "착오송금 반환 신청"
HCX_RAG_BUSINESS_FUNCTION: str | None = None

WORK_ROOT = Path("/content/kdic_rag_baseline")
EXTRACT_ROOT = WORK_ROOT / "uploaded_output"
RESULT_ROOT = WORK_ROOT / "results"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("설정 완료")
print("- Chat 모델:", HCX_CHAT_MODEL)
print("- 질문 임베딩 모델:", HCX_EMBEDDING_MODEL)
print("- Top-K:", HCX_RAG_TOP_K)
print("- 최소 유사도:", HCX_RAG_MIN_SCORE)
print("- 업무 필터:", HCX_RAG_BUSINESS_FUNCTION)


## 3. `KDIC_output.zip` 업로드 및 압축 해제

아래 셀을 실행한 뒤 기존에 사용하던 `KDIC_output.zip`을 선택합니다.

노트북은 압축 내부에서 다음 세 파일을 자동으로 찾습니다.

```text
documents.jsonl
chunks.jsonl
chunk_embeddings_hcx.jsonl
```



### 코드 해설 — ZIP 업로드와 압축 해제

두 함수의 역할이 분리되어 있습니다.

- `upload_kdic_output_zip()`: 사용자가 올린 ZIP 바이트를 Colab 파일로 저장
- `extract_kdic_output()`: ZIP 손상 여부를 확인한 후 작업 폴더에 압축 해제

실행 흐름:

```text
files.upload()
→ ZIP 파일인지 확인
→ 정확히 1개인지 확인
→ KDIC_output_uploaded.zip으로 저장
→ 기존 압축 해제 폴더 정리
→ ZIP 검사(testzip)
→ 압축 해제
```

마지막 두 줄은 함수를 정의만 하는 것이 아니라 실제로 호출합니다. 따라서 이 셀을 실행하면
즉시 업로드 창이 나타납니다. 로컬 Jupyter에서는 바로 아래 안내처럼 경로를 직접 지정해야 합니다.


In [ ]:
# 1단계: 사용자가 올린 KDIC_output ZIP을 Colab 작업 공간에 저장합니다.
def upload_kdic_output_zip() -> Path:
    """Colab 업로드 창에서 ZIP 파일 하나를 받아 로컬 경로로 저장합니다."""
    try:
        from google.colab import files
    except ImportError as error:
        raise RuntimeError(
            "이 셀은 Google Colab 업로드 기능을 사용합니다. "
            "로컬 Jupyter라면 ZIP 경로를 직접 지정하는 셀을 사용하세요."
        ) from error

    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]

    if len(zip_names) != 1:
        raise RuntimeError(
            "ZIP 파일을 정확히 1개 업로드해야 합니다. "
            f"현재 ZIP 수={len(zip_names)}"
        )

    uploaded_name = zip_names[0]
    destination = WORK_ROOT / "KDIC_output_uploaded.zip"
    destination.write_bytes(uploaded[uploaded_name])

    print("업로드 완료:", uploaded_name)
    print("저장 경로:", destination)
    return destination


def extract_kdic_output(zip_path: Path) -> Path:
    """기존 압축 해제 폴더를 지우고 ZIP 전체를 안전하게 해제합니다."""
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP 파일이 없습니다: {zip_path}")

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f"ZIP 손상 파일 발견: {bad_member}")
        archive.extractall(EXTRACT_ROOT)

    print("압축 해제 완료:", EXTRACT_ROOT)
    return EXTRACT_ROOT


KDIC_ZIP_PATH = upload_kdic_output_zip()
extract_kdic_output(KDIC_ZIP_PATH)


### 로컬 Jupyter에서 사용할 경우

Colab 업로드 창 대신 다음처럼 경로를 직접 지정할 수 있습니다.

```python
KDIC_ZIP_PATH = Path("/원하는/경로/KDIC_output.zip")
extract_kdic_output(KDIC_ZIP_PATH)
```


## 4. 핵심 산출물 자동 탐색·로드

압축 내부의 루트 폴더명이 달라도 `processed` 아래 핵심 파일을 재귀적으로 찾습니다.



### 코드 해설 — 산출물 위치 탐색과 JSONL 로드

`JSONL`은 한 줄에 JSON 객체 하나가 저장된 형식입니다. 파일 전체를 하나의 JSON 배열로
읽지 않고, 줄마다 `json.loads()`를 수행합니다.

- `find_unique_file`: 압축을 푼 폴더 아래에서 원하는 파일명을 재귀 검색합니다.
- `processed` 폴더 안의 파일을 우선하여 엉뚱한 복사본 선택을 줄입니다.
- 후보가 0개이거나 여러 개면 조용히 추측하지 않고 오류로 중단합니다.
- `load_jsonl`: 빈 줄은 건너뛰고, 잘못된 JSON이 있으면 정확한 줄 번호를 알려줍니다.
- `RESULT`: 기존 검색 함수가 기대하는 `documents`, `chunks` 구조를 유지하기 위한 묶음입니다.

여기서는 아직 검색하지 않습니다. 디스크에 있는 산출물을 파이썬 리스트로 올리는 단계입니다.


In [ ]:
# 압축 내부 경로가 달라도 파일명으로 핵심 산출물을 찾아냅니다.
def find_unique_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"압축 해제 결과에서 {filename}을 찾지 못했습니다."
        )

    # processed 폴더 내부 파일을 우선합니다.
    processed_matches = [
        path for path in matches
        if path.parent.name == "processed"
    ]
    candidates = processed_matches or matches

    if len(candidates) != 1:
        raise RuntimeError(
            f"{filename} 후보가 여러 개입니다: "
            + ", ".join(str(path) for path in candidates)
        )

    return candidates[0]


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSONL 파싱 실패: {path}, line={line_number}"
                ) from error

            if not isinstance(record, dict):
                raise TypeError(
                    f"JSONL 레코드가 객체가 아닙니다: {path}, line={line_number}"
                )
            records.append(record)

    return records


DOCUMENTS_PATH = find_unique_file(EXTRACT_ROOT, "documents.jsonl")
CHUNKS_PATH = find_unique_file(EXTRACT_ROOT, "chunks.jsonl")
EMBEDDINGS_PATH = find_unique_file(
    EXTRACT_ROOT,
    "chunk_embeddings_hcx.jsonl",
)


documents = load_jsonl(DOCUMENTS_PATH)
chunks = load_jsonl(CHUNKS_PATH)
embedding_records = load_jsonl(EMBEDDINGS_PATH)

# 기존 V4.6 검색 함수가 사용하던 구조를 유지합니다.
RESULT = {
    "documents": documents,
    "chunks": chunks,
}

print("로드 완료")
print("- documents:", len(documents), DOCUMENTS_PATH)
print("- chunks:", len(chunks), CHUNKS_PATH)
print("- embeddings:", len(embedding_records), EMBEDDINGS_PATH)


## 5. Baseline 데이터 무결성 검사

답변을 생성하기 전에 다음을 확인합니다.

- 문서 ID 중복 여부
- 청크 ID 중복 여부
- 청크와 임베딩의 1:1 연결
- 임베딩 차원 통일 여부
- 저장된 임베딩 모델이 `bge-m3`인지
- 6개 업무 값 확인



### 코드 해설 — 데이터 무결성 검사

검색을 시작하기 전에 데이터가 서로 정확히 연결되는지 검사합니다. 이 단계가 없으면
검색 점수는 나왔는데 대응하는 본문 청크가 없거나, 잘못된 청크가 답변 근거로 들어갈 수 있습니다.

검사 항목:

1. 문서·청크·임베딩 레코드의 ID가 비어 있지 않은지
2. 각각의 ID가 중복되지 않는지
3. 모든 청크에 임베딩이 있는지(`missing_embeddings`)
4. 원본 청크가 없는 임베딩이 있는지(`orphan_embeddings`)
5. 모든 임베딩 벡터의 차원이 같은지
6. 벡터에 `NaN`, 무한대 같은 계산 불가능 값이 없는지
7. 실제 데이터에 들어 있는 업무 목록이 무엇인지

`set A - set B`는 A에는 있지만 B에는 없는 항목을 찾는 집합 연산입니다. 이 노트북에서
가장 중요한 검사는 `chunk_id`를 기준으로 청크와 임베딩이 1:1 대응하는지 확인하는 부분입니다.


In [ ]:
# 검색 전에 문서-청크-임베딩 연결이 안전한지 검증합니다.
def assert_unique(values: list[str], label: str) -> None:
    duplicates = pd.Series(values).value_counts()
    duplicates = duplicates[duplicates > 1]
    if not duplicates.empty:
        raise RuntimeError(
            f"{label} 중복 발견: {duplicates.index.tolist()[:10]}"
        )


document_ids = [
    str(document.get("doc_id", "")).strip()
    for document in documents
]
chunk_ids = [
    str(chunk.get("chunk_id", "")).strip()
    for chunk in chunks
]
embedding_chunk_ids = [
    str(record.get("chunk_id", "")).strip()
    for record in embedding_records
]

if any(not value for value in document_ids):
    raise RuntimeError("빈 doc_id가 존재합니다.")
if any(not value for value in chunk_ids):
    raise RuntimeError("빈 chunk_id가 존재합니다.")
if any(not value for value in embedding_chunk_ids):
    raise RuntimeError("임베딩에 빈 chunk_id가 존재합니다.")

assert_unique(document_ids, "doc_id")
assert_unique(chunk_ids, "chunk_id")
assert_unique(embedding_chunk_ids, "embedding chunk_id")

chunk_id_set = set(chunk_ids)
embedding_chunk_id_set = set(embedding_chunk_ids)

missing_embeddings = sorted(chunk_id_set - embedding_chunk_id_set)
orphan_embeddings = sorted(embedding_chunk_id_set - chunk_id_set)

if missing_embeddings:
    raise RuntimeError(
        f"임베딩이 없는 청크가 있습니다: {missing_embeddings[:20]}"
    )
if orphan_embeddings:
    raise RuntimeError(
        f"원본 청크가 없는 임베딩이 있습니다: {orphan_embeddings[:20]}"
    )

embedding_dimensions = {
    int(record.get("dimensions", 0))
    for record in embedding_records
}
embedding_models = {
    str(record.get("model", ""))
    for record in embedding_records
}

if len(embedding_dimensions) != 1 or 0 in embedding_dimensions:
    raise RuntimeError(
        f"임베딩 차원이 일치하지 않습니다: {embedding_dimensions}"
    )

for record in embedding_records:
    vector = record.get("embedding")
    if not isinstance(vector, list):
        raise TypeError(
            f"embedding이 list가 아닙니다: {record.get('chunk_id')}"
        )
    if len(vector) != record.get("dimensions"):
        raise RuntimeError(
            f"임베딩 길이 불일치: {record.get('chunk_id')}"
        )
    if not all(math.isfinite(float(value)) for value in vector):
        raise RuntimeError(
            f"NaN 또는 무한대 임베딩 발견: {record.get('chunk_id')}"
        )

business_functions = sorted({
    str(chunk.get("business_function", "")).strip()
    for chunk in chunks
    if str(chunk.get("business_function", "")).strip()
})

print("무결성 검사 통과")
print("- 청크-임베딩 연결:", len(chunk_ids), "건")
print("- 임베딩 차원:", embedding_dimensions)
print("- 저장 임베딩 모델:", embedding_models)
print("- 업무 목록:")
for value in business_functions:
    print("  -", value)


## 6. HCX API 키와 클라이언트 설정

Colab 왼쪽 **Secrets**에 다음 이름으로 API 키를 등록합니다.

```text
HCX_API_KEY
```

키 값 앞에 `Bearer `를 붙이지 않습니다.



### 코드 해설 — API 키 읽기와 HCX 클라이언트 생성

API 키는 코드에 직접 적지 않고 다음 순서로 찾습니다.

```text
Colab Secrets의 HCX_API_KEY
→ 없으면 운영체제 환경 변수 HCX_API_KEY
→ 둘 다 없으면 오류
```

그 후 줄바꿈, `Bearer ` 접두어, 공백이 섞이지 않았는지 검사합니다. 정상 키를
`OpenAI(...)`에 전달하되 `base_url`을 HCX 주소로 설정하여 클로바 API를 호출합니다.

보안상 중요한 점:

- API 키 값을 출력하지 않고 “등록됨”만 출력합니다.
- 노트북을 팀원에게 공유할 때 키 자체가 셀이나 출력에 들어 있지 않아야 합니다.
- 팀원은 각자 자신의 Secret 또는 환경 변수에 키를 등록해야 합니다.


In [ ]:
# API 키는 코드에 직접 쓰지 않고 Secret 또는 환경 변수에서만 읽습니다.
try:
    from google.colab import userdata
except ImportError:
    userdata = None


def load_hcx_api_key() -> str:
    """Colab Secret 또는 환경 변수에서 HCX API 키를 읽고 검증합니다."""
    key = None

    if userdata is not None:
        try:
            key = userdata.get("HCX_API_KEY")
        except Exception:
            key = None

    if not key:
        key = os.environ.get("HCX_API_KEY")

    if not key:
        raise RuntimeError(
            "HCX API 키가 없습니다. Colab Secrets에 HCX_API_KEY를 "
            "등록한 뒤 이 셀을 다시 실행하세요."
        )

    key = str(key).strip()
    lines = [line.strip() for line in key.splitlines() if line.strip()]

    if len(lines) != 1:
        raise RuntimeError(
            "HCX_API_KEY는 줄바꿈 없는 API 키 하나여야 합니다."
        )

    key = lines[0]

    if key.lower().startswith("bearer "):
        raise RuntimeError(
            "HCX_API_KEY 앞에 'Bearer '를 붙이지 마세요."
        )

    if any(character.isspace() for character in key):
        raise RuntimeError(
            "HCX_API_KEY 안에 공백 또는 줄바꿈이 포함되어 있습니다."
        )

    return key


HCX_API_KEY = load_hcx_api_key()

hcx_client = OpenAI(
    api_key=HCX_API_KEY,
    base_url=HCX_BASE_URL,
    timeout=HCX_REQUEST_TIMEOUT_SECONDS,
    max_retries=HCX_MAX_RETRIES,
)

print("HCX 클라이언트 설정 완료")
print("- Base URL:", HCX_BASE_URL)
print("- Chat 모델:", HCX_CHAT_MODEL)
print("- 질문 임베딩 모델:", HCX_EMBEDDING_MODEL)
print("- API 키: 등록됨")


## 7. HCX 호출 함수

기존 V4.6 방식과 동일하게 질문 임베딩 요청에는 문자열 하나와 `encoding_format="float"`를 사용합니다.



### 코드 해설 — 채팅 API와 임베딩 API 포장 함수

이 셀은 외부 API 호출을 두 함수로 분리합니다.

- `hcx_chat_text`: 시스템 지침과 사용자 프롬프트를 보내 최종 텍스트 답변을 받습니다.
- `hcx_embed_text`: 질문 하나를 `bge-m3` 숫자 벡터 하나로 바꿉니다.

`normalize_hcx_embedding_text`는 NULL 문자와 줄바꿈 형식을 정리하고 빈 질문을 차단합니다.
임베딩 결과는 다음 조건을 확인합니다.

- 요청 1건에 벡터도 정확히 1개인지
- 벡터가 비어 있지 않은지
- `NaN` 또는 무한대가 없는지
- 저장된 청크 임베딩과 차원이 같은지

질문 벡터와 청크 벡터의 차원이 다르면 코사인 유사도를 계산할 수 없으므로 즉시 중단합니다.


In [ ]:
# 최종 답변 생성과 질문 임베딩 생성을 각각 함수로 감쌉니다.
def hcx_chat_text(
    *,
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = 500,
    temperature: float = 0.1,
) -> str:
    response = hcx_client.chat.completions.create(
        model=HCX_CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    text = response.choices[0].message.content
    if not text or not text.strip():
        raise RuntimeError("HCX 응답 본문이 비어 있습니다.")
    return text.strip()


def normalize_hcx_embedding_text(text: str) -> str:
    cleaned = (
        str(text)
        .replace("\x00", "")
        .replace("\r\n", "\n")
        .replace("\r", "\n")
        .strip()
    )

    if not cleaned:
        raise ValueError("임베딩 입력 텍스트가 비어 있습니다.")
    return cleaned


def hcx_embed_text(text: str) -> list[float]:
    """질문 문자열 한 건을 bge-m3 벡터 하나로 변환합니다."""
    cleaned_text = normalize_hcx_embedding_text(text)

    response = hcx_client.embeddings.create(
        model=HCX_EMBEDDING_MODEL,
        input=cleaned_text,
        encoding_format=HCX_EMBEDDING_ENCODING_FORMAT,
    )

    if len(response.data) != 1:
        raise RuntimeError(
            "단일 임베딩 요청의 응답 개수가 1개가 아닙니다. "
            f"응답 개수={len(response.data)}"
        )

    vector = [float(value) for value in response.data[0].embedding]

    if not vector:
        raise RuntimeError("HCX 임베딩 벡터가 비어 있습니다.")
    if not all(math.isfinite(value) for value in vector):
        raise RuntimeError(
            "HCX 임베딩 벡터에 NaN 또는 무한대가 포함되어 있습니다."
        )

    stored_dimensions = next(iter(embedding_dimensions))
    if len(vector) != stored_dimensions:
        raise RuntimeError(
            "질문 임베딩과 저장 청크 임베딩의 차원이 다릅니다: "
            f"query={len(vector)}, stored={stored_dimensions}"
        )

    return vector


print("HCX 호출 함수 준비 완료")


## 8. 기존 V4.6 Dense 검색·근거 기반 답변 함수

다음 로직은 V4.6 Baseline을 유지합니다.

```text
질문 임베딩
→ 코사인 유사도
→ 선택적 business_function 필터
→ 선택적 최소 점수 필터
→ Top-K
→ HCX-005 근거 기반 답변
→ 허용되지 않은 URL·전화번호 제거
→ 근거 청크·공식 출처 추가
```



### 코드 해설 — 검색과 근거 기반 답변의 핵심

이 셀이 현재 RAG의 핵심입니다.

#### 1. `cosine_similarity`

두 벡터의 방향이 얼마나 비슷한지 계산합니다. 값이 클수록 질문과 청크의 의미가 가깝다고
간주합니다. 두 벡터 중 하나가 영벡터면 0을 반환하여 0으로 나누는 오류를 막습니다.

#### 2. `semantic_search_hcx`

```text
질문 → 질문 임베딩 → 모든 저장 청크 벡터와 비교
→ 선택한 업무가 아니면 제외 → 최소 점수 미만 제외
→ 점수 내림차순 정렬 → 앞에서 Top-K개 반환
```

현재는 **Dense 임베딩 검색만** 사용합니다. BM25 키워드 검색, Hybrid, Reranking,
업무 자동 분류는 아직 포함되지 않았습니다.

#### 3. 허용 정보 수집·사후 필터

`_allowed_evidence_values`가 검색된 청크 안의 URL과 전화번호만 허용 목록으로 만듭니다.
`_remove_unsupported_urls_and_phones`는 LLM이 허용 목록 밖의 값을 만들면 삭제합니다.
이는 환각을 줄이는 안전장치지만, 답변 내용 전체의 사실성을 완벽히 보장하는 평가는 아닙니다.

#### 4. `generate_grounded_hcx_answer`

검색 결과가 없거나 최고 점수가 기준 미만이면 LLM을 호출하지 않고 “근거 부족” 답변을 냅니다.
근거가 있으면 청크 내용을 번호별로 묶어 프롬프트에 넣고, 낮은 temperature로 답변을 생성합니다.
마지막에는 실제 검색에 사용된 청크 ID와 공식 출처 URL을 코드가 직접 덧붙입니다.


In [ ]:
# [검색 단계] 질문과 청크 벡터의 의미 유사도를 계산합니다.
def cosine_similarity(
    vector_a: np.ndarray,
    vector_b: np.ndarray,
) -> float:
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector_a, vector_b) / denominator)


def semantic_search_hcx(
    question: str,
    *,
    top_k: int = 5,
    business_function: str | None = None,
    min_score: float | None = None,
) -> list[dict]:
    if not embedding_records:
        raise RuntimeError(
            "임베딩 결과가 없습니다. KDIC_output을 먼저 로드하세요."
        )

    query_vector = np.asarray(
        hcx_embed_text(question),
        dtype=np.float32,
    )
    chunks_by_id = {
        chunk["chunk_id"]: chunk
        for chunk in RESULT["chunks"]
    }
    scored = []

    for record in embedding_records:
        if (
            business_function
            and record.get("business_function") != business_function
        ):
            continue

        vector = np.asarray(
            record["embedding"],
            dtype=np.float32,
        )
        score = cosine_similarity(query_vector, vector)

        if min_score is not None and score < min_score:
            continue

        chunk = chunks_by_id.get(record["chunk_id"])
        if chunk:
            scored.append({"score": score, "chunk": chunk})

    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:top_k]


def _allowed_evidence_values(
    search_results: list[dict],
) -> tuple[set[str], set[str]]:
    urls: set[str] = set()
    phones: set[str] = set()
    phone_pattern = re.compile(
        r"(?:0\d{1,2}[-)]\s*)?\d{3,4}-\d{4}"
    )

    for result in search_results:
        chunk = result["chunk"]

        for value in [
            chunk.get("source_url"),
            chunk.get("official_download_url"),
        ]:
            if value:
                urls.add(value.rstrip("/"))

        for link in chunk.get("related_links", []):
            if link.get("url"):
                urls.add(link["url"].rstrip("/"))

        phones.update(
            phone_pattern.findall(chunk.get("content", ""))
        )

    return urls, phones


def _remove_unsupported_urls_and_phones(
    answer: str,
    allowed_urls: set[str],
    allowed_phones: set[str],
) -> str:
    url_pattern = re.compile(r"https?://[^\s)\]}>]+")
    phone_pattern = re.compile(
        r"(?:0\d{1,2}[-)]\s*)?\d{3,4}-\d{4}"
    )

    def url_replacer(match: re.Match) -> str:
        candidate = match.group(0).rstrip(".")
        return (
            candidate
            if candidate.rstrip("/") in allowed_urls
            else ""
        )

    def phone_replacer(match: re.Match) -> str:
        candidate = match.group(0)
        return candidate if candidate in allowed_phones else ""

    answer = url_pattern.sub(url_replacer, answer)
    answer = phone_pattern.sub(phone_replacer, answer)
    answer = re.sub(r"[ \t]+\n", "\n", answer)
    answer = re.sub(r"\n{3,}", "\n\n", answer)
    return answer.strip()


def generate_grounded_hcx_answer(
    question: str,
    search_results: list[dict],
    *,
    min_score: float = HCX_RAG_MIN_SCORE,
) -> str:
    if (
        not search_results
        or search_results[0]["score"] < min_score
    ):
        return (
            "검색된 공식 근거의 관련도가 충분하지 않아 "
            "현재 수집 데이터만으로는 정확하게 답할 수 없습니다."
        )

    evidence_blocks = []

    for rank, result in enumerate(search_results, start=1):
        chunk = result["chunk"]
        evidence_blocks.append("\n".join([
            f"[근거 {rank}]",
            f"문서 ID: {chunk.get('document_id')}",
            f"청크 ID: {chunk.get('chunk_id')}",
            f"업무: {chunk.get('business_function')}",
            f"출처: {chunk.get('source_url')}",
            "내용:",
            chunk.get("content", ""),
        ]))

    allowed_urls, allowed_phones = _allowed_evidence_values(
        search_results
    )
    evidence_text = "\n\n".join(evidence_blocks)

    raw_answer = hcx_chat_text(
        system_prompt=(
            "당신은 예금보험공사 공식 문서 기반 RAG 답변기입니다. "
            "제공된 근거에 명시된 정보만 사용하세요. "
            "근거에 없는 금액, 기한, 조건, 기관, URL, 전화번호를 "
            "만들지 마세요. 근거가 부족하면 확인할 수 없다고 답하세요. "
            "출처 목록은 작성하지 마세요."
        ),
        user_prompt=f"""
[사용자 질문]
{question}

[검색된 공식 근거]
{evidence_text}

[답변에 사용 가능한 URL]
{sorted(allowed_urls)}

[답변에 사용 가능한 전화번호]
{sorted(allowed_phones)}

한국어로 답변하세요. URL과 전화번호는 위 허용 목록에 있는 값만 그대로 사용할 수 있습니다.
""".strip(),
        max_tokens=900,
        temperature=0.0,
    )

    answer = _remove_unsupported_urls_and_phones(
        raw_answer,
        allowed_urls,
        allowed_phones,
    )

    used_chunks = [
        result["chunk"].get("chunk_id")
        for result in search_results
    ]
    source_urls = list(dict.fromkeys(
        result["chunk"].get("source_url")
        for result in search_results
        if result["chunk"].get("source_url")
    ))

    appendix = "\n\n근거 청크: " + ", ".join(used_chunks)
    if source_urls:
        appendix += "\n공식 출처:\n" + "\n".join(
            f"- {url}" for url in source_urls
        )

    return answer + appendix


print("Baseline 검색·답변 함수 준비 완료")


## 9. 질문 실행 함수

`run_kdic_rag()`는 검색 결과 표, 최종 답변, 저장된 JSON 경로를 반환합니다.



### 코드 해설 — 질문 1건을 끝까지 처리하는 파이프라인

`run_kdic_rag()`가 질문 한 건에 대한 전체 작업을 지휘합니다.

```text
입력값 검증
→ semantic_search_hcx()
→ 화면용 검색 결과 표 생성
→ generate_grounded_hcx_answer()
→ 질문·설정·검색결과·답변을 payload로 묶음
→ 선택적으로 JSON 저장
→ payload 반환
```

- `build_search_result_rows`: 내부 청크 객체에서 검토에 필요한 필드만 골라 표 형태로 만듭니다.
- `utc_now_iso`: 결과 생성 시각을 시간대에 덜 의존하는 UTC ISO 형식으로 남깁니다.
- `save_result=True`: 질문마다 결과를 JSON으로 보존합니다.
- 반환되는 `payload`는 이후 평가 코드나 UI가 재사용하기 쉬운 구조입니다.

주의: 파일명은 초 단위 시각이므로 같은 초에 여러 질문을 저장하면 덮어쓸 가능성이 있습니다.
학습용 초기 버전에서는 괜찮지만 서비스화할 때 UUID나 밀리초를 추가하는 편이 안전합니다.


In [ ]:
# 질문 한 건을 검색부터 답변 저장까지 연결하는 실행 파이프라인입니다.
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def build_search_result_rows(
    search_results: list[dict],
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []

    for rank, result in enumerate(search_results, start=1):
        chunk = result["chunk"]
        rows.append({
            "rank": rank,
            "score": round(float(result["score"]), 6),
            "document_id": chunk.get("document_id"),
            "chunk_id": chunk.get("chunk_id"),
            "business_function": chunk.get("business_function"),
            "chunk_type": chunk.get("chunk_type"),
            "title": chunk.get("title"),
            "section_title": chunk.get("section_title"),
            "source_url": chunk.get("source_url"),
            "preview": chunk.get("content", "")[:250],
        })

    return rows


def run_kdic_rag(
    question: str,
    *,
    top_k: int = HCX_RAG_TOP_K,
    business_function: str | None = HCX_RAG_BUSINESS_FUNCTION,
    min_score: float = HCX_RAG_MIN_SCORE,
    save_result: bool = True,
) -> dict[str, Any]:
    question = str(question).strip()
    if not question:
        raise ValueError("질문이 비어 있습니다.")
    if top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다.")
    if business_function and business_function not in business_functions:
        raise ValueError(
            "존재하지 않는 업무 필터입니다. "
            f"허용값={business_functions}"
        )

    search_results = semantic_search_hcx(
        question,
        top_k=top_k,
        business_function=business_function,
        min_score=min_score,
    )
    result_rows = build_search_result_rows(search_results)
    result_df = pd.DataFrame(result_rows)

    print("질문:", question)
    print("업무 필터:", business_function)
    print("Top-K:", top_k)
    print("최소 유사도:", min_score)
    print("\n검색 결과")
    display(result_df)

    answer = generate_grounded_hcx_answer(
        question,
        search_results,
        min_score=min_score,
    )

    print("\nHCX 답변\n")
    print(answer)

    payload = {
        "question": question,
        "business_function_filter": business_function,
        "minimum_score": min_score,
        "chat_model": HCX_CHAT_MODEL,
        "embedding_model": HCX_EMBEDDING_MODEL,
        "top_k": top_k,
        "retrieved": result_rows,
        "answer": answer,
        "generated_at": utc_now_iso(),
    }

    if save_result:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        result_path = RESULT_ROOT / f"hcx_rag_{timestamp}.json"
        result_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        payload["result_path"] = str(result_path)
        print("\n저장:", result_path)

    return payload


print("질문 실행 함수 준비 완료")


## 10. 대화형 질문 입력 및 답변 생성

이제 코드의 `QUESTION` 변수에 질문을 미리 작성하지 않습니다.

아래 셀을 한 번 실행하면 입력창이 반복해서 나타납니다.

```text
사용자 질문 > 카카오뱅크로 돈을 잘못 보냈는데 돌려받을 수 있나요?
```

질문을 입력할 때마다 기존과 동일하게 다음 결과가 출력됩니다.

- 입력한 질문과 검색 설정
- 검색된 Top-K 청크 표
- HCX 답변
- 답변에 사용한 근거 청크
- 공식 출처 URL
- 개별 결과 JSON 저장 경로

대화를 끝내려면 `종료`, `끝`, `exit`, `quit` 중 하나를 입력합니다.



### 코드 해설 — 반복 입력형 챗봇

이 셀은 웹 UI가 아니라 터미널 입력창을 반복 사용하는 간단한 대화 루프입니다.

- `BUSINESS_FUNCTION_FILTER=None`: 업무를 자동 분류하지 않고 6개 업무 전체 검색
- `chat_history`: 현재 실행 세션에서 성공한 결과를 순서대로 보관
- `while True`: 사용자가 종료 명령을 입력할 때까지 반복
- 빈 질문은 다시 입력받고, API 오류가 나도 전체 프로그램을 끝내지 않고 다음 질문으로 진행
- 성공한 결과는 `chat_history`와 전역 `rag_output`에 저장

중요한 한계: 이름은 `chat_history`이지만 이전 대화 내용을 다음 질문의 검색·답변 프롬프트에
넣지는 않습니다. 즉, 현재 구현은 “여러 질문을 연속 실행”할 뿐 **문맥을 기억하는 멀티턴 챗봇은 아닙니다.**


In [ ]:
# 사용자가 종료 명령을 입력할 때까지 질문을 반복해서 처리합니다.
# None이면 6개 업무 전체에서 검색합니다.
# 특정 업무만 검색하려면 아래 값을 6개 업무 중 하나로 변경할 수 있습니다.
BUSINESS_FUNCTION_FILTER = None

# 기존 V4.6 Baseline 검색 설정을 그대로 사용합니다.
TOP_K = 5
MIN_SCORE = 0.30

EXIT_COMMANDS = {"종료", "끝", "exit", "quit", "q"}
chat_history: list[dict[str, Any]] = []
rag_output: dict[str, Any] | None = None


def run_kdic_chat() -> list[dict[str, Any]]:
    """입력창에서 질문을 반복해서 받아 기존 RAG 출력 형식으로 실행합니다."""
    print("=" * 72)
    print("예금보험공사 문서 기반 RAG 질의응답")
    print("질문을 직접 입력하세요.")
    print("종료 명령어: 종료 / 끝 / exit / quit / q")
    print("=" * 72)

    while True:
        try:
            question = input("\n사용자 질문 > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n입력을 종료합니다.")
            break

        if not question:
            print("질문이 비어 있습니다. 내용을 입력해 주세요.")
            continue

        if question.lower() in EXIT_COMMANDS:
            print("대화형 질의응답을 종료합니다.")
            break

        print("\n" + "-" * 72)

        try:
            output = run_kdic_rag(
                question,
                top_k=TOP_K,
                business_function=BUSINESS_FUNCTION_FILTER,
                min_score=MIN_SCORE,
            )
        except Exception as error:
            print("\n질문 처리 중 오류가 발생했습니다.")
            print(f"- 오류 유형: {type(error).__name__}")
            print(f"- 오류 내용: {error}")
            print("다른 질문을 입력하거나 설정과 API 연결 상태를 확인해 주세요.")
            continue

        chat_history.append(output)

        # 직전 결과를 노트북 전역 변수에 저장해
        # 다음 다운로드 셀에서 그대로 사용할 수 있게 합니다.
        globals()["rag_output"] = output

        print("-" * 72)
        print(f"현재 세션 처리 질문 수: {len(chat_history)}")

    print(f"\n세션 종료 — 총 {len(chat_history)}개 질문 처리")
    return chat_history


chat_history = run_kdic_chat()


## 11. 선택: 마지막 질문 결과 JSON 다운로드

대화형 실행을 종료한 뒤 아래 셀을 실행하면 **마지막으로 성공한 질문의 결과 JSON**을 다운로드합니다.

각 질문의 결과는 실행할 때마다 `results` 폴더에 개별 JSON으로 저장됩니다.



### 코드 해설 — 마지막 결과 다운로드

대화형 셀에서 마지막으로 성공한 결과가 `rag_output`에 남아 있습니다. 이 셀은 그 객체의
`result_path`를 찾아 Colab 다운로드를 시작합니다.

- Colab이면 `files.download(...)`를 호출합니다.
- 로컬 Jupyter이면 자동 다운로드 대신 저장 경로를 출력합니다.
- 질문이 한 번도 성공하지 않았다면 다운로드할 파일이 없다는 안내를 표시합니다.

모든 질문의 결과는 이미 `results` 폴더에 각각 저장되지만, 이 셀은 그중 마지막 결과 한 건만
편하게 내려받는 기능입니다.


In [ ]:
# 마지막으로 성공한 질문 결과 JSON을 내려받습니다.
try:
    from google.colab import files

    if not rag_output:
        raise RuntimeError(
            "다운로드할 결과가 없습니다. 먼저 대화형 질문 셀에서 "
            "질문을 한 건 이상 성공적으로 실행하세요."
        )

    result_path = rag_output.get("result_path")
    if not result_path:
        raise RuntimeError("마지막 결과의 저장 경로가 없습니다.")

    files.download(result_path)
except ImportError:
    if rag_output and rag_output.get("result_path"):
        print(
            "로컬 Jupyter에서는 다음 파일을 직접 확인하세요:",
            rag_output["result_path"],
        )
    else:
        print("저장된 마지막 결과가 없습니다.")


## 현재 노트북의 범위

포함:

- 기존 V4.6 문서·청크·임베딩 업로드 및 로드
- 데이터 연결 검사
- 입력창 기반 반복 질문
- 질문마다 1건 임베딩
- 기존 Dense 코사인 유사도 검색
- 기존 HCX-005 Baseline 답변 생성
- 허용 URL·전화번호 제한
- 검색 결과 표 출력
- 근거 청크·공식 출처 출력
- 질문별 결과 JSON 저장
- 마지막 질문 결과 다운로드

제외:

- 크롤링
- HTML 파싱
- 첨부파일 텍스트 추출
- 청킹
- 전체 청크 임베딩 재생성
- BM25
- Hybrid Retrieval
- Reranking
- 업무 자동 분류
- Parent-Child 문맥 확장
- 평가 데이터셋 자동 실행
- 별도 웹 UI 또는 API 서버

> 이번 변경은 **질문 입력 방식만 사용자 친화적으로 개선**한 것입니다. 검색과 답변 생성 Baseline 로직 및 출력 구조는 기존 V4.6을 유지합니다.
